In [1]:
!pip install -q pandas plotly panel

In [2]:
%%writefile app.py
import pandas as pd
import plotly.express as px
import panel as pn

pn.extension('plotly', sizing_mode="stretch_width")

# 1. Carregamento e Preparação dos Dados
@pn.cache
def load_data():
    url = "https://raw.githubusercontent.com/wcota/covid19br/master/cases-brazil-states.csv"
    df = pd.read_csv(url, usecols=['date', 'state', 'newCases', 'newDeaths'])
    df = df[df['state'] != 'TOTAL'].copy()

    coords = {
        'AC': (-9.02, -70.81), 'AL': (-9.57, -36.78), 'AM': (-3.41, -65.85), 'AP': (1.41, -51.77),
        'BA': (-12.96, -38.51), 'CE': (-5.20, -39.53), 'DF': (-15.78, -47.92), 'ES': (-19.18, -40.30),
        'GO': (-15.82, -49.83), 'MA': (-5.42, -45.44), 'MG': (-18.51, -44.55), 'MS': (-20.77, -54.78),
        'MT': (-12.68, -56.92), 'PA': (-1.99, -54.93), 'PB': (-7.23, -36.78), 'PE': (-8.28, -35.03),
        'PI': (-7.71, -42.72), 'PR': (-25.25, -52.02), 'RJ': (-22.84, -43.15), 'RN': (-5.81, -36.59),
        'RO': (-11.50, -63.58), 'RR': (2.73, -61.32), 'RS': (-30.03, -51.23), 'SC': (-27.24, -50.21),
        'SE': (-10.57, -37.38), 'SP': (-23.55, -46.63), 'TO': (-10.18, -48.33)
    }

    regioes = {
        'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
        'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
        'BA': 'Nordeste', 'PE': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste',
        'PB': 'Nordeste', 'RN': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'PI': 'Nordeste',
        'AM': 'Norte', 'PA': 'Norte', 'AP': 'Norte', 'RR': 'Norte', 'RO': 'Norte', 'AC': 'Norte', 'TO': 'Norte',
        'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste'
    }

    df['date'] = pd.to_datetime(df['date'])
    df['ano'] = df['date'].dt.year
    df['mes_ano'] = df['date'].dt.to_period('M').astype(str)
    df['regiao'] = df['state'].map(regioes)
    df['lat'] = df['state'].apply(lambda x: coords.get(x, (0,0))[0])
    df['lon'] = df['state'].apply(lambda x: coords.get(x, (0,0))[1])

    df_monthly = df.groupby(['ano', 'mes_ano', 'regiao', 'state', 'lat', 'lon'], as_index=False).agg({
        'newCases': 'sum',
        'newDeaths': 'sum'
    })
    return df_monthly

df_monthly = load_data()

# 2. Widgets para os Filtros (Filtro por Ano e por Região)
anos_opcoes = ['Todos'] + sorted(list(df_monthly['ano'].unique()))
regioes_opcoes = ['Todas'] + sorted(list(df_monthly['regiao'].dropna().unique()))

select_ano = pn.widgets.Select(name='Ano', options=anos_opcoes, value='Todos')
select_regiao = pn.widgets.Select(name='Região', options=regioes_opcoes, value='Todas')

# 3. Função Reativa para Gerar Visualizações
@pn.depends(select_ano.param.value, select_regiao.param.value)
def update_dashboard(ano, regiao):
    df_f = df_monthly.copy()
    if ano != 'Todos':
        df_f = df_f[df_f['ano'] == ano]
    if regiao != 'Todas':
        df_f = df_f[df_f['regiao'] == regiao]

    if df_f.empty:
        return pn.pane.Markdown("### ⚠️ Nenhum dado encontrado para o filtro selecionado.")

    # KPIs
    total_casos = df_f['newCases'].sum()
    total_obitos = df_f['newDeaths'].sum()

    kpis = pn.Row(
        pn.indicators.Number(name='Total de Novos Casos', value=total_casos, format='{value:,.0f}', colors=[(1, '#1f77b4')]),
        pn.indicators.Number(name='Total de Óbitos', value=total_obitos, format='{value:,.0f}', colors=[(1, '#d62728')])
    )

    # 1. Tabela
    df_tabela = df_f.groupby(['regiao', 'state'], as_index=False).agg({
        'newCases': 'sum', 'newDeaths': 'sum'
    }).rename(columns={'regiao': 'Região', 'state': 'UF', 'newCases': 'Total Casos', 'newDeaths': 'Total Óbitos'})

    tabela_widget = pn.widgets.DataFrame(df_tabela.head(15), height=250, width=500)

    # 2. Série Temporal
    df_time = df_f.groupby('mes_ano', as_index=False)['newCases'].sum()
    fig_line = px.line(
        df_time, x='mes_ano', y='newCases',
        title='1. Série Temporal: Evolução Mensal de Novos Casos',
        labels={'mes_ano': 'Mês/Ano', 'newCases': 'Novos Casos'},
        markers=True, template='plotly_white'
    )
    fig_line.update_traces(line_color='#1f77b4', line_width=3)
    fig_line.update_layout(xaxis_tickangle=-45, height=350)

    # 3. Mapa
    df_mapa = df_f.groupby(['state', 'lat', 'lon', 'regiao'], as_index=False)['newCases'].sum()
    fig_map = px.scatter_geo(
        df_mapa, lat='lat', lon='lon', size='newCases', color='newCases',
        hover_name='state', hover_data={'regiao': True, 'newCases': ':,', 'lat': False, 'lon': False},
        size_max=30, projection="mercator", color_continuous_scale="Reds",
        title="2. Visualização Geográfica: Concentração de Casos por Estado"
    )
    fig_map.update_geos(
        scope='south america', center=dict(lat=-14.235, lon=-51.925),
        projection_scale=3.8, showcountries=True, showland=True, landcolor="#f8f9fa"
    )
    fig_map.update_layout(height=400, margin=dict(l=0, r=0, t=30, b=0))

    return pn.Column(
        kpis,
        pn.Row(pn.Column("### 📋 Tabela Resumida", tabela_widget), pn.pane.Plotly(fig_line)),
        pn.pane.Plotly(fig_map)
    )

# Template do Dashboard
template = pn.template.FastListTemplate(
    title='📊 Dashboard Interativo de COVID-19 - Projeto Integrador',
    sidebar=[select_ano, select_regiao],
    main=[update_dashboard]
)

template.servable()

Writing app.py


In [ ]:
import urllib

# 1. Inicia o servidor Panel em segundo plano na porta 5006
!panel serve app.py --port 5006 --allow-websocket-origin=* &>/dev/null &

# 2. Pega o IP público necessário para liberar o acesso inicial ao link
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

print("================================================================")
print(f"🔑 SENHA DE ACESSO AO LINK (Endpoint IP): {ip}")
print("================================================================")
print("Aguarde o link do LocalTunnel abaixo carregar...")

# 3. Cria o link público usando LocalTunnel
!npx localtunnel --port 5006

🔑 SENHA DE ACESSO AO LINK (Endpoint IP): 136.114.191.170
Aguarde o link do LocalTunnel abaixo carregar...
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸your url is: https://forty-dragons-care.loca.lt
